<a href="https://colab.research.google.com/github/KP-365/Skinrash-detection/blob/main/models/BruteForcelolipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from datasets import load_dataset
import numpy as np
import itertools

SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 8
BATCH_SIZE = 32
IMG_SIZE = 224
QUICK_EPOCHS_HEAD = 15   # capped short for the search phase
QUICK_EPOCHS_FT = 7

clean = load_dataset("eceunal/bug-bite-images-hf")
labels = clean["train"].features["label"].names

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class BugBiteDataset(Dataset):
    def __init__(self, hf_split, transform):
        self.data = hf_split
        self.transform = transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        ex = self.data[idx]
        img = ex["image"].convert("RGB")
        return self.transform(img), ex["label"]

train_loader = DataLoader(BugBiteDataset(clean["train"], train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(BugBiteDataset(clean["validation"], eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

def build_model(dropout_p):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    for param in model.features.parameters():
        param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(nn.Dropout(p=dropout_p), nn.Linear(in_features, NUM_CLASSES))
    return model.to(DEVICE)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    correct, total = 0, 0
    for imgs, targets in loader:
        imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        correct += (outputs.argmax(1) == targets).sum().item()
        total += imgs.size(0)
    return correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, targets in loader:
            imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
            outputs = model(imgs)
            correct += (outputs.argmax(1) == targets).sum().item()
            total += imgs.size(0)
    return correct / total

criterion = nn.CrossEntropyLoss()

# --- Grid to search ---
dropout_options = [0.3, 0.4, 0.5]
lr_options = [1e-5, 3e-5, 1e-4]
unfreeze_options = [1, 3]  # how many final blocks to unfreeze in phase 2

grid = list(itertools.product(dropout_options, lr_options, unfreeze_options))
print(f"Total combinations to test: {len(grid)}")

search_results = []

for dropout_p, ft_lr, unfreeze_n in grid:
    print(f"\n--- Testing: dropout={dropout_p}, ft_lr={ft_lr}, unfreeze_last={unfreeze_n} ---")

    model = build_model(dropout_p)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

    # Quick phase 1
    for epoch in range(QUICK_EPOCHS_HEAD):
        train_epoch(model, train_loader, optimizer, criterion)
    val_acc_p1 = eval_epoch(model, val_loader, criterion)

    # Quick phase 2
    for param in model.features[-unfreeze_n:].parameters():
        param.requires_grad = True
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=ft_lr)

    for epoch in range(QUICK_EPOCHS_FT):
        train_epoch(model, train_loader, optimizer, criterion)
    val_acc_p2 = eval_epoch(model, val_loader, criterion)

    print(f"  Phase 1 val_acc: {val_acc_p1:.4f} | Phase 2 val_acc: {val_acc_p2:.4f}")

    search_results.append({
        "dropout": dropout_p, "ft_lr": ft_lr, "unfreeze_last": unfreeze_n,
        "val_acc_phase1": val_acc_p1, "val_acc_phase2": val_acc_p2
    })

# --- Results, sorted best first ---
search_results.sort(key=lambda x: x["val_acc_phase2"], reverse=True)

print(f"\n\n{'='*70}")
print("GRID SEARCH RESULTS (sorted by phase 2 val accuracy)")
print(f"{'='*70}")
print(f"{'Dropout':<10} {'FT LR':<10} {'Unfreeze':<10} {'P1 val_acc':<12} {'P2 val_acc':<12}")
for r in search_results:
    print(f"{r['dropout']:<10} {r['ft_lr']:<10} {r['unfreeze_last']:<10} {r['val_acc_phase1']:<12.4f} {r['val_acc_phase2']:<12.4f}")

best = search_results[0]
print(f"\nBest config: dropout={best['dropout']}, ft_lr={best['ft_lr']}, unfreeze_last={best['unfreeze_last']}")
print(f"Best val_acc: {best['val_acc_phase2']:.4f}")

README.md:   0%|          | 0.00/764 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 17.5MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  918kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating validation split:   0%|          | 0/106 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/53 [00:00<?, ? examples/s]

Total combinations to test: 18

--- Testing: dropout=0.3, ft_lr=1e-05, unfreeze_last=1 ---
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 123MB/s] 


  Phase 1 val_acc: 0.5377 | Phase 2 val_acc: 0.4811

--- Testing: dropout=0.3, ft_lr=1e-05, unfreeze_last=3 ---
  Phase 1 val_acc: 0.4340 | Phase 2 val_acc: 0.5660

--- Testing: dropout=0.3, ft_lr=3e-05, unfreeze_last=1 ---
  Phase 1 val_acc: 0.5189 | Phase 2 val_acc: 0.5189

--- Testing: dropout=0.3, ft_lr=3e-05, unfreeze_last=3 ---
  Phase 1 val_acc: 0.5472 | Phase 2 val_acc: 0.6132

--- Testing: dropout=0.3, ft_lr=0.0001, unfreeze_last=1 ---
  Phase 1 val_acc: 0.5094 | Phase 2 val_acc: 0.5472

--- Testing: dropout=0.3, ft_lr=0.0001, unfreeze_last=3 ---
  Phase 1 val_acc: 0.4906 | Phase 2 val_acc: 0.6604

--- Testing: dropout=0.4, ft_lr=1e-05, unfreeze_last=1 ---
  Phase 1 val_acc: 0.4906 | Phase 2 val_acc: 0.4906

--- Testing: dropout=0.4, ft_lr=1e-05, unfreeze_last=3 ---
  Phase 1 val_acc: 0.4717 | Phase 2 val_acc: 0.5094

--- Testing: dropout=0.4, ft_lr=3e-05, unfreeze_last=1 ---
  Phase 1 val_acc: 0.4906 | Phase 2 val_acc: 0.4906

--- Testing: dropout=0.4, ft_lr=3e-05, unfreeze_l